# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabsajjad19/ML-capstone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For this search intelligence data contract, the **unit of analysis** is a `daily (page, keyword) pair`. This means each row represents the performance of a specific keyword on a specific page for a single day.

The **time window** covers `historical data from January 1, 2023, to December 31, 2023`, providing a full year of data for trend analysis and model training.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import pandas as pd
import numpy as np

# Simulate a dummy DataFrame for demonstration
# In a real scenario, you would load your actual data here

start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2023-12-31')
dates = pd.date_range(start=start_date, end=end_date, freq='D')

keywords = ['how to make coffee', 'best coffee machine', 'coffee shop near me', 'coffee beans types']
pages = ['/blog/coffee-guide', '/products/espresso-makers', '/local/coffee-locations', '/learn/coffee-varieties']

data_rows = []
for date in dates:
    for page in pages:
        for keyword in keywords:
            clicks = np.random.randint(0, 1000)
            impressions = np.random.randint(clicks, 5000)
            position = np.random.uniform(1.0, 30.0)
            ctr = clicks / impressions if impressions > 0 else 0
            data_rows.append([date, page, keyword, clicks, impressions, position, ctr])

df = pd.DataFrame(data_rows, columns=['date', 'page', 'keyword', 'clicks', 'impressions', 'avg_position', 'ctr'])

# Display the first few rows to confirm the grain
print("First 5 rows of the DataFrame:")
display(df.head())

# Verify the time window
print(f"\nData covers from {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")

# Verify the unit of analysis by checking for duplicates in the primary key (date, page, keyword)
duplicates = df.duplicated(subset=['date', 'page', 'keyword']).sum()
print(f"\nNumber of duplicate (date, page, keyword) combinations: {duplicates}")
if duplicates == 0:
    print("Unit of analysis is correctly defined: each (date, page, keyword) combination is unique.")
else:
    print("Warning: Duplicates found, unit of analysis might not be unique.")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Based on the simulated data and common search intelligence metrics, here's a classification of fields:

### Features
These are the input variables used to predict the label or understand search performance.
- `date`: The specific day of the observation.
- `page`: The URL path of the page being analyzed.
- `keyword`: The search query for which the page appeared.
- `impressions`: The number of times the page appeared in search results for the keyword.
- `avg_position`: The average ranking of the page for the keyword.

### Label
This is the target variable we are trying to predict or optimize.
- `clicks`: The number of times users clicked on the page for the keyword.

### Context
These fields provide additional information that might be useful for understanding the data but are not directly used as features or labels in a simple model.
- `ctr`: Click-through rate (clicks / impressions). This is often a calculated metric and can be a feature or derived from features and label.

### Excluded
These fields are not included in the analysis for specific reasons.
- (None in this dummy example, but typically might include PII, internal IDs, or highly correlated redundant features that are removed during feature engineering.)



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Inspect the columns and their data types
print("DataFrame Info:")
df.info()

print("\nDataFrame Columns:")
for col in df.columns:
    print(f"- {col}")

# You can manually verify against your proposed categories, e.g.:
# features = ['date', 'page', 'keyword', 'impressions', 'avg_position']
# label = 'clicks'
# context = ['ctr']
# excluded = []

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Grain Verification: Uniqueness of (date, page, keyword)
We established that the unit of analysis is a daily (page, keyword) pair. This query verifies that each such combination is unique in the dataset, confirming the grain.

### Counts: Total Number of Rows
This simply confirms the total size of our dataset.

### Missing Values: Check for Nulls per Column
Ensuring data quality by identifying any columns with missing values that might need imputation or special handling.

### Windows: Min and Max Dates
Re-confirms the date range to ensure it aligns with the expected time window and no unexpected outliers exist.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# 1. Grain Verification: Check for duplicate (date, page, keyword) combinations
duplicate_rows = df[df.duplicated(subset=['date', 'page', 'keyword'], keep=False)]
print(f"Number of duplicate (date, page, keyword) combinations: {len(duplicate_rows)}")
if len(duplicate_rows) == 0:
    print("✅ Grain verified: Each (date, page, keyword) combination is unique.")
else:
    print("❌ Grain warning: Duplicate (date, page, keyword) combinations found.")
    display(duplicate_rows.head())

# 2. Counts: Total number of rows
total_rows = len(df)
print(f"\nTotal number of rows in the dataset: {total_rows}")

# 3. Missing values: Check for nulls per column
missing_values = df.isnull().sum()
print("\nMissing values per column:")
print(missing_values)
if missing_values.sum() == 0:
    print("✅ No missing values found.")
else:
    print("❌ Missing values detected in one or more columns.")

# 4. Windows: Min and Max Dates
min_date = df['date'].min()
max_date = df['date'].max()
print(f"\nDate window: From {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
expected_min = pd.to_datetime('2023-01-01')
expected_max = pd.to_datetime('2023-12-31')

if min_date == expected_min and max_date == expected_max:
    print("✅ Date window matches the expected range.")
else:
    print("❌ Date window does not match the expected range.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### What this data can never tell you:

1.  **User Intent Beyond Query:** While we have keywords, the data doesn't reveal the full intent or context of the user's search. For example, a search for 'coffee' could be for a recipe, a shop, or a product.
2.  **Off-Platform Search Behavior:** This data only captures interactions within the search engine and doesn't account for users who find information through other channels (social media, direct navigation, etc.).
3.  **Changes in User Preferences Over Time (Granularity):** While we have daily data, macro-level shifts in user preferences or broader market trends might be hard to discern without additional external data or longer time horizons.
4.  **Impact of External Events:** The data doesn't inherently contain information about external events (e.g., holidays, news, competitor campaigns) that might influence search behavior, making it harder to explain sudden spikes or drops.
5.  **Long-Term Impact of Content Changes:** While daily data is available, understanding the sustained, long-term impact of a content update or website redesign might require more sophisticated causal inference models or A/B testing, which raw search data alone can't provide.
6.  **Reasons for No Clicks:** If a page has impressions but no clicks, the data doesn't tell us *why*. Was the title unappealing? Was the snippet irrelevant? Was the page too far down the results?


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# This section is primarily for qualitative analysis and discussion.
# No code is directly required here to 'verify' data limits, as they represent what is NOT in the data.
# However, one might run queries to confirm the *absence* of certain data points if they were expected.

print("This cell is intentionally left blank as 'Data Limits' are primarily conceptual and discussed in the markdown above.")
print("Queries in this section would typically confirm the *lack* of certain attributes or the scope of the available data.")

# Example: Verify no PII is present (conceptual check as we don't have real data)
# assert not any(col in df.columns for col in ['user_id', 'ip_address']), "Warning: PII potentially present!"

# Example: Check if external event data is present (it won't be in this dummy data)
# if 'holiday_flag' not in df.columns:
#     print("Confirmed: No explicit 'holiday_flag' or external event data in the DataFrame.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*Note: The last checkbox ('Committed to my repo') needs to be manually checked by the user after committing the notebook.*